# Lesson 1 — Variables & Data Types (JavaScript)

This lesson teaches the JavaScript essentials you will use immediately to build **ESM gamified cognitive tasks** (short tasks that run in a phone/browser).

In this project, you will write JavaScript in a Jupyter notebook, then run it in a separate browser window using Phaser.


## Learning objectives
hello

By the end of this lesson you can:

- Declare variables with `const` and `let` (and explain why we avoid `var`).
- Identify and use the key data types we need for cognitive tasks: number, string, boolean, null/undefined, object, array.
- Predict and fix common string–number mixing bugs (type coercion).
- Explain the difference between notebook (Deno) variables and browser (Phaser) variables in this workflow.
- Use variables to store task state (trial index, timestamps, reaction time).

## Big idea: you are working in two "worlds"

1) **Notebook runtime (Deno)**: runs your notebook cells and the `play(taskCode)` bridge.
2) **Browser runtime (Phaser)**: runs the code string inside `shell.html` (Phaser + DOM + `window`).

A variable defined in the notebook is **not automatically available** inside your Phaser code unless you inject its value into the `taskCode` string.

## How code runs in this course

You will work with **two runtimes**:

1. **Notebook runtime (Deno)**: where you run cells and call `play(taskCode)`.
2. **Browser runtime (Phaser)**: where the game/task actually runs in `shell.html`.

Before running any Phaser code, start a static file server on port 8080 (in a terminal):

`deno run --allow-net --allow-read https://deno.land/std@0.190.0/http/file_server.ts --port 8080`


In [ ]:
// TEACHER SETUP: The magic bridge (Notebook runtime: Deno)
// This builds a public URL to shell.html and injects your Phaser code via a URL parameter.
function play(taskCode) {
    const name = Deno.env.get("CODESPACE_NAME");
    const domain = Deno.env.get("GITHUB_CODESPACES_PORT_FORWARDING_DOMAIN");

    if (!name || !domain) {
        return "ERROR: Are you running this in a GitHub Codespace?";
    }

    const baseUrl = `https://${name}-8080.${domain}`;
    const finalUrl = `${baseUrl}/shell.html?code=${encodeURIComponent(taskCode)}`;

    Deno.jupyter.display({
        "text/html": `
            <div style="padding: 15px; border: 2px solid #4A90E2; border-radius: 8px; background: #f0f7ff; font-family: sans-serif;">
                <p style="margin: 0 0 10px 0; color: #333;"><b>Phaser Stage Ready</b></p>
                <a href="${finalUrl}" target="_blank" style="display: inline-block; padding: 8px 16px; background: #4A90E2; color: white; text-decoration: none; border-radius: 4px; font-weight: bold;">Open Big Screen Window</a>
            </div>
        `
    }, { raw: true });
}


## 1) Variables: `const` vs `let`

Think of variables as **task memory**. Some information stays fixed (configuration), and other information changes every trial (state).

- Use **`const`** for values that should not be reassigned (task settings, config objects).
- Use **`let`** for values that change while the task runs (trial counters, timers, state flags).


In [ ]:
// Notebook runtime example (runs in Deno)
const stimulusDurationMs = 800;
let trialIndex = 0;

trialIndex = trialIndex + 1;

({ stimulusDurationMs, trialIndex })


## 2) Data types you will use in cognitive tasks

In this project you will mostly use:

- **Number**: timing in milliseconds, positions, scores (e.g., `rtMs = 412`).
- **String**: labels, instructions, conditions (e.g., `condition = 'go'`).
- **Boolean**: state flags (e.g., `trialActive = true`).
- **Object**: grouped settings and configs (e.g., `taskConfig = { stimulusDurationMs: 800 }`).
- **Array**: lists of trials or stimuli (e.g., `trials = [{...}, {...}]`).

Two special values to know:
- **`null`**: intentionally empty (e.g., “no response yet”).
- **`undefined`**: missing / not set (often a bug).


In [ ]:
// Quick type checks (Notebook runtime)
const condition = 'go';
const responded = false;
const taskConfig = { stimulusDurationMs: 800, condition };
const trials = [{ condition: 'go' }, { condition: 'no-go' }];
let responseKey = null;

[
  typeof condition,
  typeof responded,
  typeof taskConfig,
  Array.isArray(trials),
  responseKey
]


## Type coercion (common bug in timing + logging)

JavaScript sometimes converts types automatically. This is convenient until it breaks timing math.

- If `rtMs` is a string like `'500'`, then `'500' + 20` becomes `'50020'` (wrong).
- Fix by converting explicitly: `Number(rtMs)` before doing math.

## Template literal safety (important in this project)

Your Phaser code is stored inside a **template literal** in the notebook. Example:

```javascript
const taskCode = `
  // Phaser code here
`;
play(taskCode);
```

That means you must avoid using backticks inside the embedded Phaser code, or you will break the string.

If you need dynamic text inside Phaser, use string concatenation (e.g., `'RT: ' + rtMs`) instead of nested template literals.

## 3) Runnable demo: a tiny reaction-time (RT) task

This mini task demonstrates how variables and data types show up in real cognitive tasks:
- numbers for timestamps and RT
- strings for responses
- booleans for state
- objects for configuration

Run the cell below, then click **Open Big Screen Window**.


In [ ]:
// Student code (Browser runtime via shell.html)
// NOTE: The code below runs in the browser, not in Deno.
const rtDemo = `
// If the notebook re-runs this code, destroy the previous game to avoid duplicate canvases.
if (window.__lesson1ReactionGame) {
  window.__lesson1ReactionGame.destroy(true);
}

// 1) CONFIG OBJECT (demonstrates: object, number, string, boolean)
const taskConfig = {
  stimulusDurationMs: 1500, // number
  promptText: "Press SPACE or TAP as soon as you see GO!", // string
  allowEarlyResponse: false // boolean
};

// 2) STATE VARIABLES (demonstrates: boolean, number|null, string|null)
let trialActive = false;      // boolean
let startTimeMs = 0;          // number
let responded = false;        // boolean
let rtMs = null;              // number | null
let responseKey = null;       // string | null

// Use high-resolution timer when available; fallback for older environments.
const nowMs = () =>
  (typeof performance !== "undefined" && typeof performance.now === "function")
    ? performance.now()
    : Date.now();

class ReactionScene extends Phaser.Scene {
  constructor() {
    super("ReactionScene");
  }

  create() {
    const centerX = this.cameras.main.width / 2;

    this.add.text(centerX, 60, "Reaction Time Demo", {
      fontFamily: "Arial",
      fontSize: "32px",
      color: "#ffffff"
    }).setOrigin(0.5);

    this.add.text(centerX, 120, taskConfig.promptText, {
      fontFamily: "Arial",
      fontSize: "20px",
      color: "#ffffaa",
      align: "center",
      wordWrap: { width: 700 }
    }).setOrigin(0.5);

    const statusText = this.add.text(centerX, 220, "Get ready...", {
      fontFamily: "Arial",
      fontSize: "28px",
      color: "#aaddff"
    }).setOrigin(0.5);

    const feedbackText = this.add.text(centerX, 300, "", {
      fontFamily: "Arial",
      fontSize: "22px",
      color: "#ffffff",
      align: "center"
    }).setOrigin(0.5);

    const startTrial = () => {
      trialActive = false;
      responded = false;
      rtMs = null;
      responseKey = null;
      startTimeMs = 0;

      statusText.setText("Wait for GO...");
      statusText.setColor("#aaddff");
      feedbackText.setText("");

      const waitMs = Phaser.Math.Between(1000, 2500);

      this.time.delayedCall(waitMs, () => {
        trialActive = true;
        startTimeMs = nowMs();

        statusText.setText("GO!");
        statusText.setColor("#88ff88");

        this.time.delayedCall(taskConfig.stimulusDurationMs, () => {
          if (trialActive && !responded) {
            trialActive = false;
            feedbackText.setText("Too slow. No response detected.");
            statusText.setText("Try again...");
            statusText.setColor("#ffaaaa");
            this.time.delayedCall(1200, startTrial);
          }
        });
      });
    };

    const handleResponse = (inputSource) => {
      if (responded) return;

      if (!trialActive) {
        if (!taskConfig.allowEarlyResponse) {
          feedbackText.setText("Too early! Wait for GO.");
          statusText.setText("Resetting trial...");
          statusText.setColor("#ffaaaa");
          responded = true;
          this.time.delayedCall(900, startTrial);
          return;
        }

        responded = true;
        responseKey = inputSource;
        rtMs = null;
        feedbackText.setText("Early response accepted (" + responseKey + ").");
        statusText.setText("Starting next trial...");
        statusText.setColor("#ffdd88");
        this.time.delayedCall(900, startTrial);
        return;
      }

      responded = true;
      trialActive = false;
      responseKey = inputSource;
      rtMs = nowMs() - startTimeMs;

      const rtRounded = Math.round(rtMs);
      feedbackText.setText("RT: " + rtRounded + " ms\nInput: " + responseKey);
      statusText.setText("Nice! Next trial soon...");
      statusText.setColor("#88ff88");

      this.time.delayedCall(1200, startTrial);
    };

    // Keyboard response: SPACE key
    this.input.keyboard.on("keydown-SPACE", () => handleResponse("space"));

    // Pointer response: mouse click / touch tap
    this.input.on("pointerdown", () => handleResponse("pointer"));

    startTrial();
  }
}

const gameConfig = {
  type: Phaser.AUTO,
  parent: "phaser-game",
  width: 800,
  height: 500,
  backgroundColor: "#1d1d2a",
  scene: ReactionScene
};

window.__lesson1ReactionGame = new Phaser.Game(gameConfig);
`;

play(rtDemo);

## Practice exercise: 10-trial RT scaffold

Goal: build a simple 10-trial reaction-time task scaffold using Lesson 1 ideas:

- `const taskConfig` object for settings (does not change)
- `const trials = [...]` as an **array of objects** (10 trials)
- `let trialIndex` as changing state
- `rtMsByTrial` array storing numbers (ms) or `null` for missed trials

Success criteria:
- exactly 10 trial objects, each with a string `conditionLabel`
- task advances trials and stops after trial 10
- results printed on screen and logged in console

In [ ]:
// Exercise starter code (Browser runtime via shell.html)
// Fill in the TODOs inside the string, then run play(exerciseCode).
const exerciseCode = `
// Lesson 1 Exercise: 10-trial RT scaffold (runs in the browser)

// If the notebook re-runs this code, destroy the previous game to avoid duplicate canvases.
if (window.__lesson1ExerciseGame) {
  window.__lesson1ExerciseGame.destroy(true);
}

// 1) Task settings belong in a const object (configuration does not change)
const taskConfig = {
  responseDeadlineMs: 1500,
  itiMs: 400,
  nTrials: 10
};

// 2) Trials belong in an array of objects (each object holds per-trial data)
// TODO: Fill this with exactly 10 objects.
// Each object MUST include a string condition label.
const trials = [
  // Example objects (replace / expand to total 10):
  // { conditionLabel: 'left',  stimulusText: 'LEFT',  correctKey: 'ArrowLeft'  },
  // { conditionLabel: 'right', stimulusText: 'RIGHT', correctKey: 'ArrowRight' },
];

// 3) State changes during the task, so use let
let trialIndex = 0;

// 4) Store RT results in an array (numbers or null)
const rtMsByTrial = [];

function makeScene() {
  return {
    create: function () {
      const cx = this.cameras.main.centerX;
      const cy = this.cameras.main.centerY;

      const headerText = this.add.text(20, 20, '', {
        fontFamily: 'sans-serif',
        fontSize: '20px',
        color: '#ffffff',
      });

      const stimulusText = this.add.text(cx, cy, '', {
        fontFamily: 'sans-serif',
        fontSize: '72px',
        color: '#ffffff',
      }).setOrigin(0.5);

      const footerText = this.add.text(20, 560, 'Use ArrowLeft / ArrowRight', {
        fontFamily: 'sans-serif',
        fontSize: '18px',
        color: '#cccccc',
      });

      let trialStartMs = 0;
      let trialActive = false;

      const endTask = () => {
        stimulusText.setText('DONE');
        headerText.setText('Task complete');

        console.log('rtMsByTrial:', rtMsByTrial);
        footerText.setText('rtMsByTrial = ' + JSON.stringify(rtMsByTrial));
      };

      const startTrial = () => {
        if (trialIndex >= trials.length) {
          endTask();
          return;
        }

        const trial = trials[trialIndex];

        headerText.setText(
          'Trial ' + (trialIndex + 1) + ' / ' + trials.length + '   condition=' + trial.conditionLabel
        );

        stimulusText.setText(trial.stimulusText);

        trialActive = true;
        trialStartMs = performance.now();

        const onKeyDown = (event) => {
          if (!trialActive) return;

          // TODO: only accept ArrowLeft / ArrowRight; ignore other keys
          // if (event.key !== 'ArrowLeft' && event.key !== 'ArrowRight') return;

          trialActive = false;
          this.input.keyboard.off('keydown', onKeyDown);

          const rtMs = Math.round(performance.now() - trialStartMs);

          // TODO: store rtMs in rtMsByTrial at the correct index
          // rtMsByTrial[trialIndex] = rtMs;

          stimulusText.setText('');
          trialIndex = trialIndex + 1;

          this.time.delayedCall(taskConfig.itiMs, startTrial, [], this);
        };

        this.input.keyboard.on('keydown', onKeyDown);

        // TODO: handle missed trials (no response before deadline)
        // Use this.time.delayedCall(taskConfig.responseDeadlineMs, ...)
        // If still trialActive at deadline:
        // - set trialActive = false
        // - remove listener
        // - store null for that trial
        // - clear stimulus, advance after ITI
      };

      // Optional helpful validation
      if (trials.length !== taskConfig.nTrials) {
        headerText.setText('ERROR: trials must contain ' + taskConfig.nTrials + ' objects');
        footerText.setText('Fix trials[] first, then re-run the cell.');
        return;
      }

      this.time.delayedCall(300, startTrial, [], this);
    },
  };
}

const config = {
  type: Phaser.AUTO,
  width: 800,
  height: 600,
  backgroundColor: '#222222',
  parent: 'phaser-game',
  scene: makeScene(),
};

window.__lesson1ExerciseGame = new Phaser.Game(config);
`;

// play(exerciseCode);  // Uncomment after you complete the TODOs

## Common pitfalls

- **Deno vs Browser**: `Deno.env` exists in the notebook, but not inside Phaser code in the browser.
- **Quoting**: your Phaser code is inside a template literal string. Be careful with backticks and quotes.
- **String + number**: `'100' + 20` becomes `'10020'`. Convert strings to numbers when needed.
- **Uninitialized variables**: `undefined` often means you forgot to set a value.


## Quick checks

1. Give one example of a value that should be `const` in a cognitive task, and one that should be `let`.
2. What is the difference between `null` and `undefined` in an RT task?
3. If `rtMs = '450'` (a string), what is `'450' + 50` vs `Number('450') + 50`?
4. Why can't Phaser code automatically access notebook variables?

## Reflection (2–3 sentences)

Which data type do you expect to use most in your task (numbers, strings, objects, arrays), and why?
What is one quoting mistake you might make when building `taskCode`, and how would you notice it?